In [6]:
import pandas as pd

In [7]:
df = pd.read_csv("../data/processed/clean_book_summaries.csv")

In [8]:
df.head()

,Title,Author,Genres,Summary
0,Animal Farm,George Orwell,"['Roman à clef', 'Satire', ""Children's literat...","Old Major, the old boar on the Manor Farm, ca..."
1,A Clockwork Orange,Anthony Burgess,"['Science Fiction', 'Novella', 'Speculative fi...","Alex, a teenager living in near-future Englan..."
2,The Plague,Albert Camus,"['Existentialism', 'Fiction', 'Absurdist ficti...",The text of The Plague is divided into five p...
3,An Enquiry Concerning Human Understanding,David Hume,['Uncategorized'],The argument of the Enquiry proceeds by a ser...
4,A Fire Upon the Deep,Vernor Vinge,"['Hard science fiction', 'Science Fiction', 'S...",The novel posits that space around the Milky ...


In [9]:
# import ast

# def create_readable_meaning(row):
#     try:
#         if isinstance(row['Genres'], str):
#             genres_list = ast.literal_eval(row['Genres'])
#         else:
#             genres_list = row['Genres']
#     except:
#         genres_list = ["Uncategorized"]
        
    
#     genres_str = ", ".join(genres_list)
    
#     return f"Title: {row['Title']}. Genres: {genres_str}. Summary: {row['Summary']}"

# df['Meaning'] = df.apply(create_readable_meaning, axis=1)
# df.head()

In [41]:
df.sample(20)

,Title,Author,Genres,Summary
3513,Sweet Women Lie,Loren D. Estleman,"['Crime Fiction', 'Mystery', 'Fiction']",In this episode Walker's ex-wife has a new bo...
16272,Boyracers,Alan Bissett,['Bildungsroman'],The narrative centres on 16 year-old Falkirk ...
6101,Mydnight's Hero,Joe Dever,"['Gamebook', ""Children's literature""]",The King of Siyen has been assassinated. Prin...
6594,Where Angels Fear,Rebecca Levene,['Speculative fiction'],"Bernice's home planet of Dellah, once a place..."
14677,The Book of Basketball: The NBA According to t...,Bill Simmons,['Sports'],"From The Wall Street Journal: :""In what passe..."
13186,Cavedweller,NaN,['Uncategorized'],"Delia Byrd is a native of Cayro, Georgia and ..."
4571,Sharpe's Eagle,Bernard Cornwell,"['Historical fiction', 'Fiction', 'Historical ...",The following is a description of the key plo...
16030,What I Loved,Siri Hustvedt,['Uncategorized'],What I Loved opens with a painting of a naked...
2983,Snow Falling on Cedars,David Guterson,"['Fiction', 'Novel']",Set on the fictional San Piedro Island in the...
3752,Dersu Uzala,V. K. Arsenʹev,['Uncategorized'],Arsenyev's book tells of his travels in the U...


In [11]:
from sentence_transformers import SentenceTransformer

/home/shivam/Projects/Books4U/server-model/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 708.12it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
print("Generating embeddings .....")
embeddings = model.encode(df['Summary'].to_list(), show_progress_bar=True)

print("Embeddings shape ", embeddings.shape)

Generating embeddings .....


Batches: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 518/518 [05:27<00:00,  1.58it/s]


Embeddings shape  (16559, 384)


In [14]:
import numpy as np
# np.save("../data/processed/book_embeddings.npy", embeddings)

In [15]:
from sklearn.metrics.pairwise import cosine_similarity
def search_books(query, top_n=5):
    query_vector = model.encode([query])
    similarities = cosine_similarity(query_vector, embeddings)[0]
    top_indices = np.argsort(similarities)[-top_n:][::-1]
    results = df.iloc[top_indices].copy()
    results['similarity_score'] = similarities[top_indices]
    
    return results[['Title', 'Author', 'Genres', 'similarity_score']]

In [16]:
search_books("time loop story and philosophy")

,Title,Author,Genres,similarity_score
4498,My Pretty Pony,Stephen King,['Uncategorized'],0.548500
14705,Tunnel Through Time,Lester del Rey,['Uncategorized'],0.546300
5035,Up the Line,Robert Silverberg,"['Science Fiction', 'Novel', 'Speculative fict...",0.539627
6469,The Last Resort,Paul Leonard,['Uncategorized'],0.535266
2580,"""Repent, Harlequin!"" Said the Ticktockman",Harlan Ellison,['Fiction'],0.520293


In [17]:
search_books("moral dilemma involving artificial intelligence")

,Title,Author,Genres,similarity_score
11314,Evil Genius,Catherine Jinks,"['Science Fiction', ""Children's literature"", '...",0.449589
1698,Tik-Tok,John Sladek,"['Science Fiction', 'Speculative fiction']",0.428119
15567,The Moral Landscape,Sam Harris,['Sociology'],0.415269
5358,The Star Fraction,Ken MacLeod,"['Science Fiction', 'Speculative fiction', 'Fi...",0.403572
1130,Isaac Asimov's Caliban,Roger MacBride Allen,"['Science Fiction', 'Speculative fiction']",0.398662


In [18]:
search_books("slow burn philosophical sci-fi")

,Title,Author,Genres,similarity_score
10527,The Flames: A Fantasy,Olaf Stapledon,"['Science Fiction', 'Novel']",0.485922
4153,The Bull's Hour,Ivan Yefremov,['Science Fiction'],0.454718
7158,Alma Cogan,Gordon Burn,"['Fiction', 'Novel']",0.444283
1174,Inferno,Jerry Pournelle,"['Science Fiction', 'Speculative fiction', 'Fi...",0.434051
9646,Babylon 5: The Passing of the Techno-Mages - I...,Jeanne Cavelos,['Science Fiction'],0.420931


In [19]:
search_books("fantasy focused on political strategy instead of battles")

,Title,Author,Genres,similarity_score
6985,The 33 Strategies of War,Robert Greene,"['Self-help', 'Psychology', 'Business', 'Milit...",0.601471
2680,End Zone,Don DeLillo,"['Speculative fiction', 'Fiction', 'Novel']",0.521958
6820,Tactics of Mistake,Gordon R. Dickson,"['Science Fiction', 'Speculative fiction', 'Fi...",0.520646
1669,The Killer Angels: A Novel of the Civil War,Michael Shaara,"['Historical fiction', 'Fiction', 'War novel',...",0.503674
14721,Grunts!,NaN,['Fantasy'],0.472116


In [20]:
search_books("romance with tragic undertones")

,Title,Author,Genres,similarity_score
7814,A Very Private Life,Michael Frayn,"['Science Fiction', 'Speculative fiction', 'Fa...",0.554389
16407,Valentines,Ólafur Jóhann Ólafsson,['Uncategorized'],0.552857
8278,Imre: A Memorandum,Edward Irenaeus Prime-Stevenson,['Novel'],0.547633
2515,Toll the Hounds,Steven Erikson,"['Speculative fiction', 'Fantasy', 'Novel']",0.528174
5350,The White Hotel,D. M. Thomas,"['Speculative fiction', 'Fantasy']",0.520619


In [21]:
search_books("disabled protagonist overcoming challenges")

,Title,Author,Genres,similarity_score
12022,Strange Life of Ivan Osokin,P. D. Ouspensky,['Uncategorized'],0.518930
10971,Blink,Ted Dekker,"['Thriller', 'Mystery', 'Novel', 'Fiction', 'S...",0.431844
12962,The Painted Man,NaN,['Fantasy'],0.426529
7007,Wizard of the Pigeons,Robin Hobb,"['Speculative fiction', 'Fantasy']",0.426166
784,The Hero with a Thousand Faces,Joseph Campbell,"['Psychology', 'Fiction', 'Philosophy', 'Socio...",0.425893


In [22]:
search_books("teacher mentoring troubled student")

,Title,Author,Genres,similarity_score
6453,There's a Boy in the Girls' Bathroom,Louis Sachar,"[""Children's literature""]",0.420141
14343,The Rehearsal,Eleanor Catton,['Novel'],0.400403
10883,The Bully: A Discussion and Activity Story,NaN,['Uncategorized'],0.379575
14246,On the Jellicoe Road,Melina Marchetta,['Young adult literature'],0.375378
15214,The Creature in the Teacher,Christopher Pike,"['Speculative fiction', ""Children's literature...",0.374349


In [23]:
search_books("lonely but beautiful")

,Title,Author,Genres,similarity_score
9935,Cloud Boy,NaN,"[""Children's literature""]",0.477326
9180,The Tokaido Road,Lucia St. Clair Robson,"['Fiction', 'Historical novel']",0.418036
9014,Who will Comfort Toffle?,Tove Jansson,['Picture book'],0.364042
13306,I Am Wings,Ralph Fletcher,['Uncategorized'],0.360528
3026,The Lonely Doll,Dare Wright,"[""Children's literature""]",0.356469


In [24]:
search_books("existential story")

,Title,Author,Genres,similarity_score
12108,A Single Man,Christopher Isherwood,['Gay novel'],0.513493
2752,Lost at Sea,Bryan Lee O'Malley,['Uncategorized'],0.480110
7389,Prayers to Broken Stones,Dan Simmons,"['Horror', 'Science Fiction', 'Speculative fic...",0.478436
8848,Who Made Stevie Crye?,Michael Bishop,"['Speculative fiction', 'Horror']",0.475781
11777,The Kin of Ata are Waiting for You,NaN,['Uncategorized'],0.466973


In [25]:
search_books("hopeful emotional journey")

,Title,Author,Genres,similarity_score
16250,Jamayah: Adventures on the Path of Return,T.L. Orcutt,"['Psychological novel', 'Adventure', 'Adventur...",0.462555
15415,A Happy Healthy You,NaN,['Uncategorized'],0.448712
11777,The Kin of Ata are Waiting for You,NaN,['Uncategorized'],0.428636
15720,The Empathic Civilization,Jeremy Rifkin,['Uncategorized'],0.419152
14454,The Sending,Isobelle Carmody,"['Science Fiction', 'Alternate history', 'Post...",0.408434


In [26]:
search_books("identity theft")

,Title,Author,Genres,similarity_score
5340,Shock,Robin Cook,"['Novel', 'Science Fiction', 'Suspense', 'Roma...",0.481976
5486,Might As Well Be Dead,Rex Stout,"['Mystery', 'Detective fiction', 'Fiction', 'S...",0.415175
12661,Ke Keno Kibhabe,Qazi Anwar Hussain,['Uncategorized'],0.402156
9931,Agaton Sax And the Diamond Thieves,NaN,['Uncategorized'],0.396439
2008,My Name is Legion,Roger Zelazny,"['Science Fiction', 'Short story']",0.380512


In [27]:
search_books("deep but not depressing")

,Title,Author,Genres,similarity_score
10613,The Twenty-Second Day,Muhammad Aladdin,['Novel'],0.387946
11744,Grief: a Novel,Andrew Holleran,['Novel'],0.379401
13531,Paint It Black: A Novel,Janet Fitch,['Novel'],0.365768
7814,A Very Private Life,Michael Frayn,"['Science Fiction', 'Speculative fiction', 'Fa...",0.362238
5322,Deep Water,Patricia Highsmith,"['Crime Fiction', 'Fiction', 'Suspense']",0.361431


In [28]:
search_books("dark academia")

,Title,Author,Genres,similarity_score
12856,The Sword of Aldones,Marion Zimmer Bradley,['Science Fiction'],0.405872
9620,Dark Gold,Christine Feehan,"['Speculative fiction', 'Fantasy', 'Fiction', ...",0.397716
3671,All Tomorrow's Parties,William Gibson,"['Cyberpunk', 'Science Fiction', 'Speculative ...",0.396215
15745,Antispin,Richard Bronson,['Thriller'],0.395791
8624,Mass Effect: Revelation,Drew Karpyshyn,"['Science Fiction', 'Speculative fiction', 'Fa...",0.386634


In [37]:
search_books("quiet emotional story")

,Title,Author,Genres,similarity_score
4620,In Dreams Begin Responsibilities,Delmore Schwartz,['Uncategorized'],0.508549
16040,The Lover's Dictionary,David Levithan,['Uncategorized'],0.475097
15964,We Who Are About To...,Joanna Russ,['Science Fiction'],0.462720
14631,Someday This Pain Will Be Useful To You,Peter Cameron,['Novel'],0.455356
7814,A Very Private Life,Michael Frayn,"['Science Fiction', 'Speculative fiction', 'Fa...",0.452810


In [30]:
search_books("ghost in a new house")

,Title,Author,Genres,similarity_score
13510,Ghosts,César Aira,['Novel'],0.573123
11602,Something Upstairs,Edward Irving Wortis,"['Mystery', 'Speculative fiction', 'Horror', '...",0.557032
12331,Death in Disguise,NaN,['Uncategorized'],0.548438
15729,The Deadly Dungeon,Ron Roy,"['Mystery', ""Children's literature""]",0.546092
1167,Maskerade,Terry Pratchett,"['Science Fiction', 'Speculative fiction', 'Fa...",0.528987


In [31]:
search_books("a man who is a doctor goes to a mental hospital to see a patient")

,Title,Author,Genres,similarity_score
6462,The Sleep of Reason,Martin Day,['Speculative fiction'],0.682587
8302,The Doctor is Sick,Anthony Burgess,"['Comic novel', 'Speculative fiction']",0.520496
15769,24 Hours,NaN,['Uncategorized'],0.498273
5876,Final Diagnosis,James White,"['Science Fiction', 'Speculative fiction', 'Fa...",0.489724
6574,Revolution Man,Paul Leonard,['Speculative fiction'],0.437156


In [35]:
search_books("Harry potter is back")

,Title,Author,Genres,similarity_score
445,Harry Potter and the Order of the Phoenix,J. K. Rowling,"['Fantasy', 'Young adult literature', 'Fiction']",0.520476
158,Harry Potter and the Philosopher's Stone,J. K. Rowling,"[""Children's literature"", 'Fantasy', 'Speculat...",0.493511
480,Harry Potter and the Chamber of Secrets,J. K. Rowling,"['Speculative fiction', 'Fantasy', 'Fiction']",0.486952
716,Harry Potter and the Prisoner of Azkaban,J. K. Rowling,"['Fantasy', 'Speculative fiction', 'Young adul...",0.458394
724,Harry Potter and the Goblet of Fire,J. K. Rowling,['Speculative fiction'],0.437845


In [46]:
df.at[15724, 'Summary']

' In the aftermath of Hurricane Katrina, a lampshade, purported to be made from the skin of a Jewish Holocaust victim, turned up in a sidewalk rummage sale in New Orleans. Purchased for $35 by Skip Henderson, the lampshade was sent to his friend Mark Jacobson, a writer living in New York. Jacobson embarked on a quest to discover the origin of the lampshade. Genetic testing showed that it was indeed made from human skin. However, because of the condition of the tanned skin, there was no way to determine the ethnic origin of the person whose skin was used, or if it was indeed a relic of the Holocaust. Over the course of the next few years Jacobson attempted to track down the origin of and examine the meaning of the lampshade, how it ended up in New Orleans, and to decide what to ultimately do with the gruesome object. Both the United States Holocaust Memorial Museum in Washington, D.C., and the Yad Vashem museum in Jerusalem, declined to take possession of the lampshade, saying that the 

In [45]:
df.sample(10)

,Title,Author,Genres,Summary
3871,Waiting for the Mahatma,R. K. Narayan,['Novel'],Sriram is a high school graduate who lives wi...
326,Coraline,Neil Gaiman,"['Fantasy', 'Speculative fiction', 'Horror', ""...","This is about a ""different"" girl named Corali..."
14569,New York,Edward Rutherfurd,['Historical novel'],The novel chronicles the birth and growth of ...
7198,Winter,Len Deighton,['Fiction'],It is a time of turmoil. A time when the horr...
11005,Lila Says,Chimo,['Novel'],Lila Says is a narrative of the protagonist's...
1305,Angels and Demons,Dan Brown,"['Thriller', 'Crime Fiction', 'Novel', 'Specul...",The plot follows Harvard symbologist Robert L...
10065,Cugel's Saga,Jack Vance,"['Dying Earth subgenre', 'Speculative fiction'...","The story picks up where the protagonist, Cug..."
11649,Camilla,NaN,['Uncategorized'],Camilla focuses on the story of the Tyrold fa...
1330,Lanark: A Life in Four Books,Alasdair Gray,"['Science Fiction', 'Fantasy', 'Fiction', 'Dys...","Lanark comprises four books, arranged in the ..."
15724,The Lampshade: A Holocaust Detective Story fro...,NaN,['Uncategorized'],"In the aftermath of Hurricane Katrina, a lamp..."
